# 📈 Gold Price Prediction
**Model:** Random Forest with time-series features  
**Dataset:** 10 years of daily OHLC gold prices (GLD ETF)  
**Target:** Next-day closing price  

**Key design decisions:**
- `TimeSeriesSplit` CV — never uses future data to predict the past
- RSI and MACD computed from scratch (no TA-Lib dependency)
- Lag features (1, 3, 7, 14, 30 day closes) as the core signal

In [ ]:
!git clone https://github.com/Aurovindhya/ml-portfolio-suite.git
%cd ml-portfolio-suite
!pip install -q scikit-learn pandas numpy matplotlib seaborn yfinance

In [ ]:
# Fetch 10 years of GLD data via yfinance
import yfinance as yf
import pandas as pd
import os

ticker = yf.Ticker('GLD')
df = ticker.history(period='10y')[['Open','High','Low','Close','Volume']]
df = df.reset_index().rename(columns={'Date': 'date'})
df.columns = df.columns.str.lower()

os.makedirs('data', exist_ok=True)
df.to_csv('data/gold_prices.csv', index=False)
print(f'GLD dataset: {df.shape[0]} trading days')
df.tail()

In [ ]:
# Visualise price + indicators
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '.')
from modules.regression.gold_prices.model import engineer_features

df_feat = engineer_features(df)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
axes[0].plot(df_feat['close'], label='Close', color='gold'); axes[0].set_title('GLD Close Price'); axes[0].legend()
axes[1].plot(df_feat['rsi_14'], label='RSI-14', color='purple'); axes[1].axhline(70, ls='--', color='r', alpha=0.5); axes[1].axhline(30, ls='--', color='g', alpha=0.5); axes[1].set_title('RSI'); axes[1].legend()
axes[2].plot(df_feat['macd'], label='MACD', color='blue'); axes[2].plot(df_feat['macd_signal'], label='Signal', color='orange'); axes[2].set_title('MACD'); axes[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
# TimeSeriesSplit — why it matters
from sklearn.model_selection import TimeSeriesSplit
import numpy as np

tscv = TimeSeriesSplit(n_splits=5)
X_dummy = np.arange(len(df_feat))

fig, axes = plt.subplots(5, 1, figsize=(14, 6))
for i, (train_idx, val_idx) in enumerate(tscv.split(X_dummy)):
    axes[i].barh(0, len(train_idx), color='steelblue', label='Train' if i==0 else '')
    axes[i].barh(0, len(val_idx), left=len(train_idx), color='tomato', label='Val' if i==0 else '')
    axes[i].set_yticks([]); axes[i].set_ylabel(f'Fold {i+1}', fontsize=8)
axes[0].legend(loc='upper right')
plt.suptitle('TimeSeriesSplit — future never leaks into training', fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
from modules.regression.gold_prices.model import train
metrics = train('data/gold_prices.csv')
print('Gold model CV metrics:', metrics)

In [ ]:
# Feature importance
import pickle
from modules.regression.gold_prices.model import FEATURE_COLS

with open('weights/gold_price.pkl', 'rb') as f:
    bundle = pickle.load(f)

importances = bundle['model'].feature_importances_
sorted_idx  = importances.argsort()[::-1]

plt.figure(figsize=(10, 5))
plt.bar([FEATURE_COLS[i] for i in sorted_idx], importances[sorted_idx], color='gold')
plt.xticks(rotation=45, ha='right')
plt.title('Random Forest Feature Importance — Gold Price')
plt.tight_layout(); plt.show()

In [ ]:
# Predict next-day price from latest row
from modules.regression.gold_prices.model import predict

latest = df_feat[FEATURE_COLS].iloc[-1].to_dict()
price, ms = predict(latest)
print(f'Predicted next-day GLD close: ${price:.2f} ({ms:.1f}ms)')
print(f'Actual last close:            ${df_feat["close"].iloc[-1]:.2f}')